# Mini-TP 2 — Actividad entregable (Sesión 2)

**Expón los metadatos de tu modelo por GraphQL y compáralo con REST.**
Individual · entrega esta semana.

## Consigna
1. Define un **esquema GraphQL** (Strawberry) con un tipo `Model` (`name`, `version`, `metrics`).
2. Una **query** que devuelva las métricas/experimentos de tu modelo.
3. Pruébalo desde **GraphiQL** y desde un **cliente Python**.
4. **Compara** la misma lectura contra tu endpoint **REST** de la Sesión 1 (llamadas y datos) y anota la diferencia.

El modelo es el de predicción de stroke del TP final de Aprendizaje de Máquina II (reentrenado localmente en `../common/train_model.py`; ver `../common/inference.py`).

## 1. Los metadatos de mi modelo

Se leen del modelo real (no son datos de ejemplo).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from common.inference import load_model

model_wrapper = load_model()
MI_MODELO = {
    "name": model_wrapper.name,
    "version": model_wrapper.version,
    "model_type": model_wrapper.data_dict["model_type"],
    "metrics": model_wrapper.metrics,
}
MI_MODELO

{'name': 'stroke_prediction_model_prod',
 'version': 1,
 'model_type': 'RandomForestClassifier',
 'metrics': {'accuracy': 0.9660668380462725,
  'precision': 0.9758403361344538,
  'recall': 0.9557613168724279,
  'f1_score': 0.9656964656964657,
  'train_observations': 7777,
  'test_observations': 1945}}

## 2. El esquema GraphQL

Tipo `Model` con `name`, `version` y `metrics` (tipado, no texto libre).

In [2]:
import strawberry
from fastapi import FastAPI
from strawberry.fastapi import GraphQLRouter

@strawberry.type
class Metrics:
    accuracy: float
    precision: float
    recall: float
    f1_score: float
    train_observations: int
    test_observations: int

@strawberry.type
class Model:
    name: str
    version: int
    model_type: str

    @strawberry.field
    def metrics(self) -> Metrics:
        m = MI_MODELO["metrics"]
        return Metrics(
            accuracy=m["accuracy"], precision=m["precision"], recall=m["recall"],
            f1_score=m["f1_score"], train_observations=m["train_observations"],
            test_observations=m["test_observations"],
        )

@strawberry.type
class Query:
    @strawberry.field
    def model(self) -> Model:
        return Model(name=MI_MODELO["name"], version=MI_MODELO["version"], model_type=MI_MODELO["model_type"])

schema = strawberry.Schema(query=Query)
app = FastAPI(title="Mini-TP 2 - metadatos por GraphQL")
app.include_router(GraphQLRouter(schema), prefix="/graphql")
print("Esquema listo")

Esquema listo


## 3. Levantar la API (en segundo plano) y consultar

GraphiQL queda disponible en `http://127.0.0.1:8010/graphql`.

In [3]:
import threading, time, uvicorn, requests

def _run():
    uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8010, log_level="warning")).run()

threading.Thread(target=_run, daemon=True).start(); time.sleep(2)

query = "{ model { name version metrics { f1Score accuracy } } }"
r = requests.post("http://127.0.0.1:8010/graphql", json={"query": query})
print(r.status_code, r.json())

200 {'data': {'model': {'name': 'stroke_prediction_model_prod', 'version': 1, 'metrics': {'f1Score': 0.9656964656964657, 'accuracy': 0.9660668380462725}}}}


## 4. Comparación con REST

Se levanta también la API REST de la Sesión 1 (mismo modelo, `/v1/model`) para comparar **la misma vista** (`name`, `version`, `f1_score`) por ambos protocolos.

In [4]:
sys.path.insert(0, str(Path.cwd().parent / "mini_tp1_rest"))
from api import app as rest_app

def _run_rest():
    uvicorn.Server(uvicorn.Config(rest_app, host="127.0.0.1", port=8011, log_level="warning")).run()

threading.Thread(target=_run_rest, daemon=True).start(); time.sleep(2)

rest_response = requests.get("http://127.0.0.1:8011/v1/model")
graphql_response = requests.post("http://127.0.0.1:8010/graphql", json={"query": query})

rest_bytes = len(rest_response.content)
graphql_bytes = len(graphql_response.content)

print("REST   :", rest_response.json())
print("bytes REST   :", rest_bytes)
print("GraphQL:", graphql_response.json())
print("bytes GraphQL:", graphql_bytes)
print(f"\nAmbos necesitaron 1 sola llamada. GraphQL trajo "
      f"{100 * (1 - graphql_bytes / rest_bytes):.0f}% menos bytes que REST para la misma vista,"
      f" porque REST siempre devuelve el modelo completo (over-fetching) y GraphQL solo"
      f" los campos pedidos.")

REST   : {'name': 'stroke_prediction_model_prod', 'version': 1, 'model_type': 'RandomForestClassifier', 'metrics': {'accuracy': 0.9660668380462725, 'precision': 0.9758403361344538, 'recall': 0.9557613168724279, 'f1_score': 0.9656964656964657, 'train_observations': 7777, 'test_observations': 1945}}
bytes REST   : 271
GraphQL: {'data': {'model': {'name': 'stroke_prediction_model_prod', 'version': 1, 'metrics': {'f1Score': 0.9656964656964657, 'accuracy': 0.9660668380462725}}}}
bytes GraphQL: 141

Ambos necesitaron 1 sola llamada. GraphQL trajo 48% menos bytes que REST para la misma vista, porque REST siempre devuelve el modelo completo (over-fetching) y GraphQL solo los campos pedidos.


**Conclusión:** con un solo recurso chico, ambos protocolos necesitan una única llamada (no hay N+1 acá). La diferencia real es que REST no tiene forma de pedir un subconjunto de campos — siempre trae el objeto completo — mientras que GraphQL deja que el cliente declare exactamente lo que necesita. Esa ventaja crece cuando el modelo tiene más metadatos (hiperparámetros, historial de runs) o cuando distintos clientes necesitan vistas distintas del mismo modelo.

## 5. (Opcional +) Linaje desde Neo4j

No implementado en esta entrega — ver `graphql_neo4j_lineage.ipynb` como base para una futura iteración.

In [5]:
# limpieza: los threads de uvicorn son daemon, se cierran solos al terminar el kernel
print("Listo.")

Listo.
